In [ ]:
# 1) stringoutputParser
# 2) jsonoutparser

In [24]:
import os
from langchain_openai import ChatOpenAI
from langchain_google_genai import ChatGoogleGenerativeAI
from dotenv import load_dotenv
from typing import TypedDict , Annotated , List , Optional
from datetime import datetime
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser , JsonOutputParser

load_dotenv()
OPENROUTER_API_KEY = os.getenv("OPENROUTER_API_KEY")
GOOGLE_API_KEY = os.getenv("GOOGLE_API_KEY")

llm_gemini = ChatGoogleGenerativeAI(model="gemini-2.0-flash" , api_key= GOOGLE_API_KEY)
llm_gemini.invoke("who is father of india")

AIMessage(content='Mahatma Gandhi is widely considered the "Father of India."', additional_kwargs={}, response_metadata={'prompt_feedback': {'block_reason': 0, 'safety_ratings': []}, 'finish_reason': 'STOP', 'safety_ratings': []}, id='run--e5950c4c-b160-4142-9ea8-ea13c4bc0202-0', usage_metadata={'input_tokens': 5, 'output_tokens': 14, 'total_tokens': 19, 'input_token_details': {'cache_read': 0}})

In [ ]:
# woking with model.content instead of stroutputParser
template1 = PromptTemplate(
    template= "write a deatailed report on the {topic}",
    input_variables= ['topic']
    )

template2 = PromptTemplate(
    template= "Give me 5 point summary for this given text: \n {text}",
    input_variables= ['text']
    )

promt1 = template1.invoke({'topic' : "cricket"})
result1 = llm_gemini.invoke(promt1)

promt2 = template2.invoke({'text' : result1.content})
final_result = llm_gemini.invoke(promt2)

print(final_result.content)

In [23]:
# woking stroutputParser clean syntax and much eaiser
template1 = PromptTemplate(
    template= "write a deatailed report on the {topic}",
    input_variables= ['topic']
    )

template2 = PromptTemplate(
    template= "Give me 5 point summary for this given text: \n {text}",
    input_variables= ['text']
    )

parser = StrOutputParser()

chain = template1 | llm_gemini | parser | template2 | llm_gemini | parser

print(chain.invoke("chess"))

Here's a 5-point summary of the provided text:

1.  **Chess is an ancient strategy game:** Originating in 6th-century India as Chaturanga, chess evolved through Persia and Europe to become the game we know today, with standardized rules and international governance by FIDE.
2.  **The game is played with specific rules and pieces:** Chess involves two players aiming to checkmate the opponent's king on an 8x8 board, utilizing pieces with unique movement capabilities and special moves like castling and en passant.
3.  **Chess requires strategic and tactical thinking:** Successful play involves understanding opening principles, middlegame strategies, endgame techniques, and tactical motifs like forks, pins, and sacrifices.
4.  **Chess notation allows for game recording and analysis:** Algebraic notation provides a standardized system for documenting moves, facilitating study and review of chess games.
5.  **Chess is culturally significant:** Chess permeates literature, art, and education, 

In [15]:
# Create Schema
from pydantic import BaseModel, Field
from typing import TypedDict

class Itenary(BaseModel):
    key_themes : Annotated[List[str] , "give me the theme that we are flowwing for this day in details"]
    day : Annotated[int , "a what number of day it is"]
    time : Annotated[str , "a time frame gap it is, e.g (8:00:00 - 2:20:20)"]
    description : str
    estimated_amount : Annotated[Optional[float] , "Give me a esctimated amt of money for this in US $"]
    
# llm = ChatOpenAI(model="openai/gpt-4o-mini", openai_api_base="https://openrouter.ai/api/v1", api_key=OPENROUTER_API_KEY) ## LLM 
structured_llm = llm_gemini.with_structured_output(schema=Itenary)
results = structured_llm.invoke("give me itenary to travel in india for for 7 days day wise")
print(results)

key_themes=['travel', 'delhi'] day=1 time='Morning' description='Arrive in Delhi and check into your hotel.' estimated_amount=100.0
